<a href="https://colab.research.google.com/github/tbruffy/financial-economics-ps5/blob/main/ps5_q2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Problem Set 5 — Problem 2

## Part (b): Persistent consumption growth

Simulate 100,000 years of consumption growth and asset returns
using the policy functions derived in part (a).

Calculate the mean, standard deviation, and first-order
autocorrelation of each requested series, and compare with Problem 1.

## Part (c): Bond yields

Derive and simulate bond yields for maturities of one to five years,
then examine how consumption-growth persistence affects their behavior.

In [1]:
import numpy as np
import pandas as pd
from scipy.optimize import brentq

# Simulation settings
T = 100_000
SEED = 42

# Parameters from Problem 2(b)
rho = 0.4
mu = 0.018
gamma = 10.0
beta = 0.99

# Target unconditional standard deviation of consumption growth
sigma_c = 0.036

# Standard deviation of the innovation in the AR(1) process
sigma = sigma_c * np.sqrt(1 - rho**2)

print(f"Persistence, rho: {rho:.2f}")
print(f"Innovation standard deviation, sigma: {sigma:.8f}")
print(
    "Implied unconditional consumption-growth standard deviation: "
    f"{sigma / np.sqrt(1 - rho**2):.8f}"
)

Persistence, rho: 0.40
Innovation standard deviation, sigma: 0.03299455
Implied unconditional consumption-growth standard deviation: 0.03600000


In [2]:
# Equation whose root determines the Campbell–Shiller coefficient
def phi_equation(phi):
    return (
        np.log(phi)
        - np.log(beta)
        - (1 - gamma) * mu
        - 0.5 * (1 - gamma)**2 * sigma**2
        / (1 - phi * rho)**2
    )


# Solve for phi within its admissible range: 0 < phi < 1
phi = brentq(
    phi_equation,
    1e-8,
    1 - 1e-8,
    xtol=1e-14
)

# Log price-dividend ratio coefficients
a = np.log(phi / (1 - phi))
b = (1 - gamma) * rho / (1 - phi * rho)

# Equity-return loading on the consumption innovation
h = 1 + phi * b

# Return intercepts
rf0 = (
    -np.log(beta)
    + gamma * mu
    - 0.5 * gamma**2 * sigma**2
)

c0 = (
    -np.log(beta)
    + gamma * mu
    - 0.5 * (h - gamma)**2 * sigma**2
)

print(f"phi = {phi:.8f}")
print(f"a   = {a:.8f}")
print(f"b   = {b:.8f}")
print(f"h   = {h:.8f}")
print(f"rf0 = {rf0:.8f}")
print(f"c0  = {c0:.8f}")

print(
    "Root equation satisfied:",
    np.isclose(phi_equation(phi), 0.0, atol=1e-10, rtol=0.0)
)

phi = 0.94333655
a   = 2.81229380
b   = -5.78159654
h   = -4.45399136
rf0 = 0.13561834
c0  = 0.07633216
Root equation satisfied: True


In [3]:
# Reproducible sequence of standard normal innovations
rng = np.random.default_rng(SEED)
epsilon = rng.standard_normal(T)

# T transitions require T + 1 states, including the initial state
x = np.empty(T + 1)

# Draw the initial state from the stationary distribution
x[0] = rng.normal(loc=0.0, scale=sigma_c)

# Simulate the centered AR(1) consumption-growth process
for t in range(T):
    x[t + 1] = rho * x[t] + sigma * epsilon[t]

# Consumption growth and log stochastic discount factor
delta_c = mu + x[1:]
m = np.log(beta) - gamma * delta_c

# Log risk-free return: determined by the current state
r_f = rf0 + gamma * rho * x[:-1]

# Log equity return under the Campbell–Shiller approximation
r_d = (
    c0
    + gamma * rho * x[:-1]
    + h * sigma * epsilon
)

# Excess log equity return
r_excess = r_d - r_f

# Collect the five requested series
data = pd.DataFrame({
    "r_f": r_f,
    "delta_c": delta_c,
    "m": m,
    "r_d": r_d,
    "r_excess": r_excess
})

print(data.head().round(6))
print(f"\nNumber of simulated years: {len(data):,}")

        r_f   delta_c         m       r_d  r_excess
0 -0.091124  0.005380 -0.063848 -0.195191 -0.104067
1  0.085137 -0.021362  0.203569  0.178685  0.093547
2 -0.021829  0.027016 -0.280211 -0.191400 -0.169571
3  0.171682  0.052640 -0.536450 -0.025827 -0.197509
4  0.274178 -0.032518  0.315125  0.501611  0.227433

Number of simulated years: 100,000


In [4]:
# Sample standard deviation
def sample_std(series):
    values = np.asarray(series, dtype=float)

    if np.all(values == values[0]):
        return 0.0

    return float(np.std(values, ddof=1))


# First-order sample autocorrelation
def acf1(series):
    values = np.asarray(series, dtype=float)

    if np.all(values == values[0]):
        return np.nan

    centered = values - values.mean()

    return float(
        np.dot(centered[1:], centered[:-1])
        / np.dot(centered, centered)
    )


# Summary statistics for the five simulated series
statistics = pd.DataFrame({
    "Mean": data.mean(),
    "Standard deviation": data.apply(sample_std),
    "First autocorrelation": data.apply(acf1)
})

print(
    statistics.to_string(
        float_format=lambda value: f"{value:.8f}"
    )
)

                Mean  Standard deviation  First autocorrelation
r_f       0.13468150          0.14455013             0.40020771
delta_c   0.01776665          0.03613721             0.40021392
m        -0.18771687          0.36137212             0.40021392
r_d       0.07601739          0.20649895            -0.26030460
r_excess -0.05866411          0.14750238             0.00129805


## Part (c): Bond yields for maturities of one to five years

In [5]:
# Bond maturities, measured in years
max_maturity = 5
maturities = np.arange(1, max_maturity + 1)

# Include maturity zero to implement the initial conditions
A = np.zeros(max_maturity + 1)
B = np.zeros(max_maturity + 1)

# Recursively calculate the bond-price coefficients
for n in maturities:
    A[n] = (
        A[n - 1]
        - np.log(beta)
        + gamma * mu
        - 0.5 * sigma**2 * (gamma + B[n - 1])**2
    )

    B[n] = rho * (gamma + B[n - 1])

# Convert bond-price coefficients into annual yield coefficients
bond_coefficients = pd.DataFrame({
    "Maturity": maturities,
    "A_n": A[1:],
    "B_n": B[1:],
    "A_n/n": A[1:] / maturities,
    "B_n/n": B[1:] / maturities
})

print(
    bond_coefficients.to_string(
        index=False,
        float_format=lambda value: f"{value:.8f}"
    )
)

 Maturity        A_n        B_n      A_n/n      B_n/n
        1 0.13561834 4.00000000 0.13561834 4.00000000
        2 0.21898195 5.60000000 0.10949098 2.80000000
        3 0.27656657 6.24000000 0.09218886 2.08000000
        4 0.32305926 6.49600000 0.08076481 1.62400000
        5 0.36499032 6.59840000 0.07299806 1.31968000


In [6]:
# Apply each bond-yield policy to the existing simulated state path
yields = pd.DataFrame({
    f"y_{n}": (A[n] + B[n] * x[:-1]) / n
    for n in maturities
})

# Use the same statistical definitions as in part (b)
yield_statistics = pd.DataFrame({
    "Mean": yields.mean(),
    "Standard deviation": yields.apply(sample_std),
    "First autocorrelation": yields.apply(acf1)
})

# Check that the one-year yield equals the risk-free return
print(
    "One-year yield matches r_f:",
    np.allclose(yields["y_1"], r_f)
)

print(f"Observations per yield: {len(yields):,}")
print()

print(
    yield_statistics.to_string(
        float_format=lambda value: f"{value:.8f}"
    )
)

One-year yield matches r_f: True
Observations per yield: 100,000

          Mean  Standard deviation  First autocorrelation
y_1 0.13468150          0.14455013             0.40020771
y_2 0.10883519          0.10118509             0.40020771
y_3 0.09170171          0.07516607             0.40020771
y_4 0.08038446          0.05868735             0.40020771
y_5 0.07268898          0.04768998             0.40020771


In [7]:
# Persistence values to compare
rho_values = [0.0, 0.2, 0.4, 0.6, 0.8]

sensitivity_rows = []

for rho_case in rho_values:

    # Keep unconditional consumption-growth volatility at 0.036
    sigma_case = sigma_c * np.sqrt(1 - rho_case**2)

    # Simulate the state using the same innovations and initial draw
    x_case = np.empty(T + 1)
    x_case[0] = x[0]

    for t in range(T):
        x_case[t + 1] = (
            rho_case * x_case[t]
            + sigma_case * epsilon[t]
        )

    # Bond-price coefficients for this persistence value
    A_case = np.zeros(max_maturity + 1)
    B_case = np.zeros(max_maturity + 1)

    for n in maturities:
        A_case[n] = (
            A_case[n - 1]
            - np.log(beta)
            + gamma * mu
            - 0.5 * sigma_case**2
            * (gamma + B_case[n - 1])**2
        )

        B_case[n] = rho_case * (gamma + B_case[n - 1])

        # Simulated yield for this maturity and persistence value
        yield_case = (
            A_case[n] + B_case[n] * x_case[:-1]
        ) / n

        sensitivity_rows.append({
            "rho": rho_case,
            "Maturity": n,
            "Mean": yield_case.mean(),
            "Standard deviation": sample_std(yield_case),
            "First autocorrelation": acf1(yield_case)
        })

# Combine results for all persistence values and maturities
sensitivity = pd.DataFrame(sensitivity_rows)

print(
    sensitivity.to_string(
        index=False,
        float_format=lambda value: f"{value:.6f}"
    )
)

     rho  Maturity     Mean  Standard deviation  First autocorrelation
0.000000         1 0.125250            0.000000                    NaN
0.000000         2 0.125250            0.000000                    NaN
0.000000         3 0.125250            0.000000                    NaN
0.000000         4 0.125250            0.000000                    NaN
0.000000         5 0.125250            0.000000                    NaN
0.200000         1 0.127467            0.072279               0.200752
0.200000         2 0.113931            0.043367               0.200752
0.200000         3 0.107416            0.029875               0.200752
0.200000         4 0.103851            0.022551               0.200752
0.200000         5 0.101663            0.018064               0.200752
0.400000         1 0.134682            0.144550               0.400208
0.400000         2 0.108835            0.101185               0.400208
0.400000         3 0.091702            0.075166               0.400208
0.4000